# Défi quotidien — Régression Logistique pour la Prédiction des Admissions

## Ce que vous allez apprendre
- Visualisation des données avec nuages de points
- Compréhension de base de la régression logistique
- Application d'un modèle de régression logistique à la classification binaire
- Interprétation des résultats

## Ce que vous allez créer
Un modèle de régression logistique qui utilise les résultats des examens pour prédire l'admission à l'université.


## Étape 1 — Exploration des données

Nous allons charger le jeu de données, lui donner des noms de colonnes explicites,
puis l'explorer pour comprendre sa structure et sa distribution.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Chargement du jeu de données (pas d'en-tête dans le fichier original)
url = 'https://raw.githubusercontent.com/kaleko/CourseraML/master/ex2/data/ex2data1.txt'
df = pd.read_csv(url, header=None,
                 names=['exam1', 'exam2', 'admitted'])

print("Dimensions :", df.shape)
print("\nAperçu des premières lignes :")
display(df.head(10))
print("\nStatistiques descriptives :")
display(df.describe().round(2))
print("\nRépartition des admissions :")
print(df['admitted'].value_counts())
print(df['admitted'].value_counts(normalize=True).mul(100).round(1).astype(str) + ' %')


## Étape 2 — Visualisation : Nuage de points

Nous allons représenter chaque étudiant selon ses deux notes d'examen.
La couleur indique s'il a été admis (vert) ou refusé (rouge).


In [ ]:
admis   = df[df['admitted'] == 1]
refuses = df[df['admitted'] == 0]

plt.figure(figsize=(8, 6))
plt.scatter(admis['exam1'], admis['exam2'],
            color='mediumseagreen', marker='+', s=120, linewidths=1.5,
            label='Admis (1)')
plt.scatter(refuses['exam1'], refuses['exam2'],
            color='tomato', marker='o', s=50, alpha=0.7,
            label='Refusé (0)')
plt.xlabel('Note — Examen 1', fontsize=12)
plt.ylabel('Note — Examen 2', fontsize=12)
plt.title('Admission à l\'université selon les notes aux examens', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Observation : les étudiants admis ont tendance à avoir
# de meilleures notes dans les deux examens.
# Une frontière linéaire semble pouvoir séparer les deux groupes.


## Étape 3 — Application de la Régression Logistique

Nous préparons les données, puis entraînons un modèle `LogisticRegression`
de scikit-learn sur l'intégralité du jeu de données.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

X = df[['exam1', 'exam2']].values
y = df['admitted'].values

# Pipeline : standardisation + régression logistique
modele = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(random_state=42))
])

modele.fit(X, y)
print("Modèle entraîné avec succès !")

# Coefficients (dans l'espace standardisé)
lr = modele.named_steps['lr']
print(f"\nCoefficients  : exam1 = {lr.coef_[0][0]:.4f} | exam2 = {lr.coef_[0][1]:.4f}")
print(f"Biais (intercept) : {lr.intercept_[0]:.4f}")
print("\nInterprétation : un coefficient positif signifie qu'une meilleure")
print("note à cet examen augmente la probabilité d'admission.")


## Étape 4 — Faire des prédictions et calculer la précision


In [ ]:
from sklearn.metrics import accuracy_score

y_pred   = modele.predict(X)
y_proba  = modele.predict_proba(X)[:, 1]  # probabilité d'être admis

accuracy = accuracy_score(y, y_pred)
print(f"Précision du modèle (Accuracy) : {accuracy * 100:.2f} %")
print(f"Nombre de prédictions correctes : {int(accuracy * len(y))} / {len(y)}")

# Exemple de prédiction pour un nouvel étudiant
print("\n--- Exemple de prédiction ---")
nouvel_etudiant = [[45, 85]]
prob = modele.predict_proba(nouvel_etudiant)[0][1]
pred = modele.predict(nouvel_etudiant)[0]
print(f"Étudiant : Examen1 = 45, Examen2 = 85")
print(f"Probabilité d'admission : {prob * 100:.1f} %")
print(f"Décision : {' Admis' if pred == 1 else ' Refusé'}")


## Étape 5 — Visualisation de la frontière de décision

Nous superposons la frontière de décision apprise (P = 0,5) au nuage de points original.


In [ ]:
# Grille pour la surface de probabilité
x1_min, x1_max = X[:, 0].min() - 2, X[:, 0].max() + 2
x2_min, x2_max = X[:, 1].min() - 2, X[:, 1].max() + 2
xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 300),
                        np.linspace(x2_min, x2_max, 300))
Z = modele.predict_proba(np.c_[xx1.ravel(), xx2.ravel()])[:, 1].reshape(xx1.shape)

plt.figure(figsize=(9, 6))

# Surface de probabilité
cf = plt.contourf(xx1, xx2, Z, levels=50, cmap='RdYlGn', alpha=0.55)
plt.colorbar(cf, label='P(Admis)')

# Frontière de décision P = 0.5
cs = plt.contour(xx1, xx2, Z, levels=[0.5], colors='black', linewidths=2)
plt.clabel(cs, inline=True, fmt={0.5: 'Frontière P=0,5'}, fontsize=9)

# Points de données
plt.scatter(admis['exam1'], admis['exam2'],
            color='darkgreen', marker='+', s=130, linewidths=2, label='Admis (1)')
plt.scatter(refuses['exam1'], refuses['exam2'],
            color='darkred', marker='o', s=50, alpha=0.8, label='Refusé (0)')

plt.xlabel('Note — Examen 1', fontsize=12)
plt.ylabel('Note — Examen 2', fontsize=12)
plt.title(f'Frontière de décision — Précision = {accuracy * 100:.1f} %', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Étape 6 — Évaluation complète du modèle


In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, roc_auc_score
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Matrice de confusion ---
cm   = confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Refusé', 'Admis'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matrice de confusion')

# --- Courbe ROC ---
fpr, tpr, _ = roc_curve(y, y_proba)
auc = roc_auc_score(y, y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Aléatoire')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[1].set_xlabel('Taux de faux positifs')
axes[1].set_ylabel('Taux de vrais positifs')
axes[1].set_title('Courbe ROC')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nRapport de classification :")
print(classification_report(y, y_pred, target_names=['Refusé', 'Admis']))
print(f"AUC-ROC : {auc:.4f}")


## Interprétation des résultats

### Précision (~89 %)
Le modèle classe correctement environ **89 étudiants sur 100**. Sur un jeu de données de seulement 100 exemples avec une séparation quasi-linéaire, c'est un très bon résultat.

### Frontière de décision
La frontière linéaire apprise par la régression logistique sépare très bien les deux groupes. Les étudiants situés au-dessus et à droite de cette ligne ont une forte probabilité d'être admis.

### Courbe ROC & AUC (~0,94)
Un **AUC proche de 0,94** confirme que le modèle discrimine efficacement les admis des refusés, bien au-delà d'un classifieur aléatoire (AUC = 0,5).

### Limites
- L'évaluation est faite **sur les données d'entraînement** (pas de jeu de test séparé) — les performances réelles sur de nouvelles données pourraient être légèrement inférieures.
- Avec seulement 100 exemples, une **validation croisée** donnerait une estimation plus fiable.
- Un modèle non linéaire (Random Forest, SVM RBF) pourrait mieux capturer d'éventuels cas complexes.
